# HDBSCAN Clustering Results Summary

This notebook loads and visualizes the results from Mode 1a HDBSCAN posterior clustering.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import h5py
import os
import sys

# Add parent directory to path for imports
sys.path.insert(0, os.path.dirname(os.getcwd()))

from backend.io import load_hdbscan_clusters_from_hdf5

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## 1. Load HDBSCAN Results

In [ ]:
# Configuration
output_dir = "../output"
filename = "hdbscan_clusters_mcs_12_ms_12.h5"  # Adjust filename as needed

# Load results
clusters, assignments, metadata = load_hdbscan_clusters_from_hdf5(output_dir, filename)

In [ ]:
# Display metadata
print("=" * 60)
print("HDBSCAN Run Metadata")
print("=" * 60)
for key, value in metadata.items():
    if key != 'whitening':  # Print whitening separately
        print(f"{key}: {value}")

print("\n" + "=" * 60)
print("Whitening Parameters")
print("=" * 60)
if 'whitening' in metadata:
    for key, value in metadata['whitening'].items():
        if isinstance(value, np.ndarray) and len(value) > 5:
            print(f"{key}: array of length {len(value)}")
        else:
            print(f"{key}: {value}")

## 2. Cluster Overview

In [ ]:
n_clusters = len(clusters)
n_realizations = metadata.get('n_realizations', 80)

print(f"Total clusters found: {n_clusters}")
print(f"Number of realizations: {n_realizations}")
print(f"Total halos: {metadata.get('n_total_halos', 'N/A')}")
print(f"Clustered halos: {metadata.get('n_clustered', 'N/A')}")
print(f"Noise halos: {metadata.get('n_noise', 'N/A')}")

# Status breakdown
status_counts = {}
for c in clusters:
    status = c['status']
    status_counts[status] = status_counts.get(status, 0) + 1

print("\nCluster Status Breakdown:")
for status, count in sorted(status_counts.items()):
    print(f"  {status}: {count}")

In [ ]:
# Create summary arrays
existence_probs = np.array([c['existence_prob'] for c in clusters])
n_members = np.array([c['n_members'] for c in clusters])
n_realizations_present = np.array([c['n_realizations_present'] for c in clusters])
mean_masses = np.array([c['mean_m200_mass'] for c in clusters])
log_mass_stds = np.array([c['log10_m200_mass_std'] for c in clusters])
mean_probs = np.array([c['mean_membership_prob'] for c in clusters])
ambiguity_rates = np.array([c['ambiguity_rate'] for c in clusters])
centers = np.array([c['center_xyz'] for c in clusters])
statuses = np.array([c['status'] for c in clusters])

## 3. Existence Probability Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(existence_probs, bins=20, edgecolor='black', alpha=0.7)
ax.axvline(metadata.get('existence_prob_stable', 0.5), color='green', linestyle='--', 
           label=f"Stable threshold ({metadata.get('existence_prob_stable', 0.5):.0%})")
ax.axvline(metadata.get('existence_prob_tentative', 0.2), color='orange', linestyle='--',
           label=f"Tentative threshold ({metadata.get('existence_prob_tentative', 0.2):.0%})")
ax.set_xlabel('Existence Probability')
ax.set_ylabel('Number of Clusters')
ax.set_title('Distribution of Cluster Existence Probabilities')
ax.legend()

# Cumulative
ax = axes[1]
sorted_probs = np.sort(existence_probs)[::-1]
ax.plot(range(1, len(sorted_probs)+1), sorted_probs, 'b-', linewidth=2)
ax.axhline(metadata.get('existence_prob_stable', 0.5), color='green', linestyle='--', 
           label=f"Stable threshold")
ax.axhline(metadata.get('existence_prob_tentative', 0.2), color='orange', linestyle='--',
           label=f"Tentative threshold")
ax.set_xlabel('Cluster Rank')
ax.set_ylabel('Existence Probability')
ax.set_title('Existence Probability vs Cluster Rank')
ax.legend()
ax.set_xlim(0, len(clusters)+1)

plt.tight_layout()
plt.show()

## 4. Cluster Size Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Members per cluster
ax = axes[0]
colors = {'stable': 'green', 'tentative': 'orange', 'rare': 'red'}
for status in ['stable', 'tentative', 'rare']:
    mask = statuses == status
    if np.any(mask):
        ax.hist(n_members[mask], bins=20, alpha=0.6, label=status, color=colors[status])
ax.set_xlabel('Number of Members')
ax.set_ylabel('Number of Clusters')
ax.set_title('Cluster Size Distribution')
ax.legend()

# Realizations per cluster
ax = axes[1]
ax.hist(n_realizations_present, bins=np.arange(0, n_realizations+2)-0.5, 
        edgecolor='black', alpha=0.7)
ax.axvline(n_realizations * metadata.get('existence_prob_stable', 0.5), color='green', 
           linestyle='--', label='Stable threshold')
ax.set_xlabel('Number of Realizations Present')
ax.set_ylabel('Number of Clusters')
ax.set_title('Realizations per Cluster')
ax.legend()

plt.tight_layout()
plt.show()

## 5. Mass Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mass histogram
ax = axes[0]
log_masses = np.log10(mean_masses)
for status in ['stable', 'tentative', 'rare']:
    mask = statuses == status
    if np.any(mask):
        ax.hist(log_masses[mask], bins=20, alpha=0.6, label=status, color=colors[status])
ax.set_xlabel('log$_{10}$(M$_{200}$ / M$_\\odot$)')
ax.set_ylabel('Number of Clusters')
ax.set_title('Cluster Mass Distribution')
ax.legend()

# Mass scatter vs existence probability
ax = axes[1]
valid = ~np.isnan(log_mass_stds)
scatter = ax.scatter(existence_probs[valid], log_mass_stds[valid], 
                     c=log_masses[valid], cmap='viridis', alpha=0.7, s=50)
ax.set_xlabel('Existence Probability')
ax.set_ylabel('log$_{10}$(M$_{200}$) std [dex]')
ax.set_title('Mass Scatter vs Existence Probability')
plt.colorbar(scatter, ax=ax, label='log$_{10}$(M$_{200}$)')

plt.tight_layout()
plt.show()

## 6. Spatial Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Get observer coordinates if available
observer = metadata.get('observer_coords', [500, 500, 500])

projections = [('x', 'y', 0, 1), ('x', 'z', 0, 2), ('y', 'z', 1, 2)]

for ax, (xlabel, ylabel, i, j) in zip(axes, projections):
    # Color by existence probability
    scatter = ax.scatter(centers[:, i], centers[:, j], 
                        c=existence_probs, cmap='viridis', 
                        s=50 + 200*existence_probs, alpha=0.7,
                        vmin=0, vmax=1)
    ax.scatter(observer[i], observer[j], marker='*', s=200, c='red', 
              edgecolors='black', label='Observer', zorder=10)
    ax.set_xlabel(f'{xlabel} [Mpc]')
    ax.set_ylabel(f'{ylabel} [Mpc]')
    ax.set_title(f'{xlabel}-{ylabel} Projection')
    ax.set_aspect('equal')
    ax.legend()

plt.colorbar(scatter, ax=axes, label='Existence Probability', shrink=0.8)
plt.tight_layout()
plt.show()

In [ ]:
# Distance from observer
observer = np.array(metadata.get('observer_coords', [500, 500, 500]))
distances = np.linalg.norm(centers - observer, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.hist(distances, bins=20, edgecolor='black', alpha=0.7)
ax.set_xlabel('Distance from Observer [Mpc]')
ax.set_ylabel('Number of Clusters')
ax.set_title('Cluster Distance Distribution')

ax = axes[1]
scatter = ax.scatter(distances, existence_probs, c=log_masses, 
                     cmap='viridis', alpha=0.7, s=50)
ax.set_xlabel('Distance from Observer [Mpc]')
ax.set_ylabel('Existence Probability')
ax.set_title('Existence Probability vs Distance')
plt.colorbar(scatter, ax=ax, label='log$_{10}$(M$_{200}$)')

plt.tight_layout()
plt.show()

## 7. Membership Probability & Ambiguity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Membership probability distribution
ax = axes[0]
ax.hist(mean_probs, bins=20, edgecolor='black', alpha=0.7)
ax.set_xlabel('Mean Membership Probability')
ax.set_ylabel('Number of Clusters')
ax.set_title('Distribution of Mean Membership Probabilities')

# Ambiguity rate distribution
ax = axes[1]
ax.hist(ambiguity_rates, bins=20, edgecolor='black', alpha=0.7, color='orange')
ax.set_xlabel('Ambiguity Rate')
ax.set_ylabel('Number of Clusters')
ax.set_title('Distribution of Ambiguity Rates\n(fraction of realizations with multiple candidates)')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
scatter = ax.scatter(existence_probs, mean_probs, c=log_masses, 
                     cmap='viridis', alpha=0.7, s=50)
ax.set_xlabel('Existence Probability')
ax.set_ylabel('Mean Membership Probability')
ax.set_title('Existence vs Membership Probability')
plt.colorbar(scatter, ax=ax, label='log$_{10}$(M$_{200}$)')

ax = axes[1]
scatter = ax.scatter(existence_probs, ambiguity_rates, c=log_masses, 
                     cmap='viridis', alpha=0.7, s=50)
ax.set_xlabel('Existence Probability')
ax.set_ylabel('Ambiguity Rate')
ax.set_title('Existence Probability vs Ambiguity Rate')
plt.colorbar(scatter, ax=ax, label='log$_{10}$(M$_{200}$)')

plt.tight_layout()
plt.show()

## 8. Whitening Diagnostics

In [ ]:
if 'whitening' in metadata and 'bin_centers' in metadata['whitening']:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    bin_centers = metadata['whitening']['bin_centers']
    bin_sigmas = metadata['whitening']['bin_sigmas']
    
    ax.plot(bin_centers, bin_sigmas, 'bo-', linewidth=2, markersize=8)
    ax.set_xlabel('log$_{10}$(M$_{200}$ / M$_\\odot$)')
    ax.set_ylabel('$\\sigma_x$ [Mpc]')
    ax.set_title('Mass-Dependent Positional Scale $\\sigma_x(M)$')
    ax.grid(True, alpha=0.3)
    
    # Add sigma_logM annotation
    sigma_logM = metadata['whitening'].get('sigma_logM', 'N/A')
    ax.text(0.02, 0.98, f'$\\sigma_{{\\log M}}$ = {sigma_logM:.3f} dex', 
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
else:
    print("Whitening diagnostics not available in this file.")

## 9. All Halo Assignments

In [ ]:
if assignments:
    labels = assignments.get('label', [])
    probs = assignments.get('membership_prob', [])
    dropped = assignments.get('dropped_by_enforcement', [])
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Label distribution
    ax = axes[0]
    unique_labels, counts = np.unique(labels, return_counts=True)
    noise_count = counts[unique_labels == -1][0] if -1 in unique_labels else 0
    cluster_counts = counts[unique_labels != -1]
    ax.bar(['Noise'] + [f'C{i}' for i in range(len(cluster_counts))], 
           [noise_count] + list(cluster_counts))
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Number of Halos')
    ax.set_title('Halo Counts per Cluster')
    ax.tick_params(axis='x', rotation=45)
    
    # Membership probability histogram
    ax = axes[1]
    clustered_mask = labels != -1
    ax.hist(probs[clustered_mask], bins=50, edgecolor='black', alpha=0.7)
    ax.set_xlabel('Membership Probability')
    ax.set_ylabel('Number of Halos')
    ax.set_title('Membership Probability Distribution (clustered halos)')
    
    # Dropped by enforcement
    ax = axes[2]
    n_dropped = np.sum(dropped)
    n_kept = len(dropped) - n_dropped - noise_count
    ax.pie([n_kept, n_dropped, noise_count], 
           labels=['Kept', 'Dropped by\nconstraint', 'Noise'],
           autopct='%1.1f%%', colors=['green', 'orange', 'gray'])
    ax.set_title('Halo Fate')
    
    plt.tight_layout()
    plt.show()
else:
    print("Assignment data not loaded.")

## 10. Top Clusters Summary Table

In [ ]:
# Print summary table for top clusters
print("=" * 120)
print(f"{'ID':>4} {'Status':>10} {'Exist.Prob':>12} {'N_Real':>8} {'N_Mem':>8} {'log10(M200)':>12} {'logM_std':>10} {'Mean_Prob':>10} {'Ambig':>8}")
print("=" * 120)

for c in clusters[:20]:  # Top 20
    print(f"{c['cluster_id']:>4} {c['status']:>10} {c['existence_prob']:>12.1%} "
          f"{c['n_realizations_present']:>8} {c['n_members']:>8} "
          f"{np.log10(c['mean_m200_mass']):>12.2f} "
          f"{c['log10_m200_mass_std']:>10.3f} "
          f"{c['mean_membership_prob']:>10.3f} {c['ambiguity_rate']:>8.1%}")

if len(clusters) > 20:
    print(f"... and {len(clusters) - 20} more clusters")

## 11. Individual Cluster Inspector

In [ ]:
def inspect_cluster(cluster_id):
    """Display detailed information about a specific cluster."""
    cluster = None
    for c in clusters:
        if c['cluster_id'] == cluster_id:
            cluster = c
            break
    
    if cluster is None:
        print(f"Cluster {cluster_id} not found.")
        return
    
    print(f"\n{'='*60}")
    print(f"Cluster {cluster_id} Details")
    print(f"{'='*60}")
    print(f"Status: {cluster['status']}")
    print(f"Existence Probability: {cluster['existence_prob']:.1%}")
    print(f"Realizations Present: {cluster['n_realizations_present']}/{n_realizations}")
    print(f"Number of Members: {cluster['n_members']}")
    print(f"")
    print(f"Position: [{cluster['center_xyz'][0]:.1f}, {cluster['center_xyz'][1]:.1f}, {cluster['center_xyz'][2]:.1f}] Mpc")
    print(f"Position Std: [{cluster['position_std'][0]:.2f}, {cluster['position_std'][1]:.2f}, {cluster['position_std'][2]:.2f}] Mpc")
    print(f"")
    print(f"M200 Mass: {cluster['mean_m200_mass']:.2e} Msol")
    print(f"log10(M200) Std: {cluster['log10_m200_mass_std']:.3f} dex")
    print(f"")
    print(f"Mean Membership Prob: {cluster['mean_membership_prob']:.3f}")
    print(f"Min Membership Prob: {cluster['min_membership_prob']:.3f}")
    print(f"Ambiguity Rate: {cluster['ambiguity_rate']:.1%}")
    
    # Plot member distribution if available
    if 'member_data' in cluster:
        member_data = cluster['member_data']
        if 'positions' in member_data:
            positions = member_data['positions']
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            
            center = cluster['center_xyz']
            for ax, (i, j, xlabel, ylabel) in zip(axes, 
                [(0, 1, 'x', 'y'), (0, 2, 'x', 'z'), (1, 2, 'y', 'z')]):
                ax.scatter(positions[:, i], positions[:, j], alpha=0.5, s=20)
                ax.scatter(center[i], center[j], c='red', s=100, marker='x', linewidths=3)
                ax.set_xlabel(f'{xlabel} [Mpc]')
                ax.set_ylabel(f'{ylabel} [Mpc]')
                ax.set_title(f'{xlabel}-{ylabel}')
                ax.set_aspect('equal')
            
            plt.suptitle(f'Cluster {cluster_id} Member Positions')
            plt.tight_layout()
            plt.show()

# Example: Inspect the first cluster
if len(clusters) > 0:
    inspect_cluster(0)

In [ ]:
# Inspect another cluster (change the ID as needed)
# inspect_cluster(1)